# 04 — Predicción de facturación diaria

## PREDICCIONES: Facturacion diaria, numero de clientes, ticket medio, ventas de un producto concreto, 

Este notebook crea un flujo de machine learning para predecir la facturación diaria utilizando variables temporales, climáticas, operativas y de negocio.

-Obs: 
Día de la semana.
Mes.
Trimestre.
Festivo (sí/no).
Fin de semana.
Vacaciones.
Facturación del día anterior.
Facturación hace 7 días.
Media de facturación de la última semana.
Número de tickets.
Número de clientes.
Ticket medio.
Temperatura o lluvia (si conseguís esos datos).

Objetivo:
- Predecir la facturación total del día
- Comparar varios modelos de regresión
- Usar validación temporal para evitar fuga de información
- Dejar una base sólida para presentar el TFM

Nota:
- Se asume que los datos depurados ya están disponibles en la carpeta `data/silver/snapshots`.
- Si no existieran ciertos archivos, puede adaptarse la carga a los nombres reales del proyecto.


# Feature Engineering: Set de variables

-temporal: dia_num
mes
es_fin_semana
es_lunes
es_festivo
dia_del_ano
semana_del_ano

-historial:
lag_1, lag_3, lag_7, lag_14, lag_30
media_movil_3, media_movil_7, media_movil_14, media_movil_30
std_7

-negocio:
num_tickets
ticket_medio
reservas_totales
reservas_grupos_grandes
ratio_walkin
cancelaciones

-clima:
temp_media
lluvia_mm
viento_max

-eventos:
es_evento
intensidad_evento
proximidad_evento

In [3]:
# 1) Importar librerías y configurar el entorno
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor

try:
    import xgboost as xgb
except Exception:
    xgb = None

try:
    import lightgbm as lgb
except Exception:
    lgb = None

try:
    import catboost as cb
except Exception:
    cb = None

# reproducibilidad
SEED = 42
np.random.seed(SEED)

# rutas del proyecto
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data'
SILVER_SNAP = DATA_DIR / 'silver' / 'snapshots'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'figures' / 'forecasting'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('talk')

print(f'Project root: {PROJECT_ROOT}')
print(f'Silver snapshots: {SILVER_SNAP}')
print(f'Results dir: {RESULTS_DIR}')

# 2) Cargar datos y revisar estructura
# Cargamos los datos depurados que ya están en Silver
# Ajusta los nombres si el proyecto usa otros archivos
for path in [
    SILVER_SNAP / 'tickets_silver.parquet',
    SILVER_SNAP / 'ventas_silver.parquet',
    SILVER_SNAP / 'festivos_silver.parquet',
    SILVER_SNAP / 'eventos_silver.parquet',
    SILVER_SNAP / 'meteo_diaria_silver.parquet',
    SILVER_SNAP / 'reservas_silver.parquet',
]:
    print(path.name, 'exists:', path.exists())

# Carga base
try:
    df_tickets = pd.read_parquet(SILVER_SNAP / 'tickets_silver.parquet')
except FileNotFoundError:
    df_tickets = pd.read_parquet(SILVER_SNAP / 'tickets_raw.parquet')

try:
    df_reservas = pd.read_parquet(SILVER_SNAP / 'reservas_silver.parquet')
except FileNotFoundError:
    df_reservas = pd.read_parquet(SILVER_SNAP / 'reservas_raw.parquet')

try:
    df_festivos = pd.read_parquet(SILVER_SNAP / 'festivos_silver.parquet')
except FileNotFoundError:
    df_festivos = pd.read_parquet(SILVER_SNAP / 'festivos_raw.parquet')

try:
    df_eventos = pd.read_parquet(SILVER_SNAP / 'eventos_silver.parquet')
except FileNotFoundError:
    df_eventos = pd.read_parquet(SILVER_SNAP / 'eventos_raw.parquet')

try:
    df_meteo = pd.read_parquet(SILVER_SNAP / 'meteo_diaria_silver.parquet')
except FileNotFoundError:
    df_meteo = pd.read_parquet(SILVER_SNAP / 'meteo_diaria_raw.parquet')

print('\nEstructura de los datasets:')
for name, df in {
    'tickets': df_tickets,
    'reservas': df_reservas,
    'festivos': df_festivos,
    'eventos': df_eventos,
    'meteo': df_meteo,
}.items():
    print(f'{name}: {df.shape}')
    print(df.dtypes.head())
    print('---')


Project root: c:\Users\CandelaGB\Desktop\TFM anita\TFM-Hosteleria-AI
Silver snapshots: c:\Users\CandelaGB\Desktop\TFM anita\TFM-Hosteleria-AI\data\silver\snapshots
Results dir: c:\Users\CandelaGB\Desktop\TFM anita\TFM-Hosteleria-AI\results\figures\forecasting
tickets_silver.parquet exists: True
ventas_silver.parquet exists: True
festivos_silver.parquet exists: True
eventos_silver.parquet exists: True
meteo_diaria_silver.parquet exists: True
reservas_silver.parquet exists: True

Estructura de los datasets:
tickets: (6709, 14)
source_file                       str
report_start           datetime64[us]
report_end             datetime64[us]
report_generated_on    datetime64[us]
terminal_start                    str
dtype: object
---
reservas: (21289, 20)
source_file                        str
reservation_datetime    datetime64[us]
created_datetime        datetime64[us]
reservation_date        datetime64[us]
reservation_time                object
dtype: object
---
festivos: (24, 4)
fecha   

In [4]:
df_tickets.head()

,source_file,report_start,report_end,report_generated_on,terminal_start,terminal_end,turn,date,document_id,document_total,receipt_count,dia_semana,es_dia_cierre,flag_outlier_importe
0,LISTA_TICKETS.xls,2025-10-01,2026-07-09,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,2025-10-02,00001TM00000001,7.00,1,3,False,False
1,LISTA_TICKETS.xls,2025-10-01,2026-07-09,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,2025-10-02,00001TM00000002,65.40,1,3,False,False
2,LISTA_TICKETS.xls,2025-10-01,2026-07-09,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,2025-10-02,00001TM00000003,8.20,1,3,False,False
3,LISTA_TICKETS.xls,2025-10-01,2026-07-09,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,2025-10-02,00001TM00000004,77.10,1,3,False,False
4,LISTA_TICKETS.xls,2025-10-01,2026-07-09,2026-07-09,1 LA ROCA POZUELO,1 LA ROCA POZUELO,T,2025-10-02,00001TM00000005,35.05,1,3,False,False


In [7]:
### 3) Explorar series temporales y distribución de la facturación

daily = (
    df_tickets.assign(date=pd.to_datetime(df_tickets['date']))
    .groupby('date', as_index=False)
    .agg(
        facturacion=('document_total', 'sum'),
        num_tickets=('document_id', 'nunique'),
        ticket_medio=('document_total', 'mean')
    )
    .sort_values('date')
    .rename(columns={'date': 'fecha'})
)

daily['fecha'] = pd.to_datetime(daily['fecha'])
print(daily.head())
print(daily.tail())
print(f'Periodo: {daily["fecha"].min()} -> {daily["fecha"].max()}')
print(f'Dias: {len(daily)}')
print(daily[['facturacion', 'num_tickets', 'ticket_medio']].describe().round(2))


       fecha  facturacion  num_tickets  ticket_medio
0 2025-10-02      1339.33           21     63.777619
1 2025-10-03      4498.50           53     84.877358
2 2025-10-04      4323.00           40    108.075000
3 2025-10-05      1866.75           16    116.671875
4 2025-10-07      1701.25           27     63.009259
         fecha  facturacion  num_tickets  ticket_medio
237 2026-07-04      3251.55           33     98.531818
238 2026-07-05      1728.35           12    144.029167
239 2026-07-07      2119.25           21    100.916667
240 2026-07-08      2183.20           31     70.425806
241 2026-07-09       788.70           12     65.725000
Periodo: 2025-10-02 00:00:00 -> 2026-07-09 00:00:00
Dias: 242
       facturacion  num_tickets  ticket_medio
count       242.00       242.00        242.00
mean       2799.55        27.72        102.20
std        1346.66        11.32         33.45
min         637.90         6.00         42.25
25%        1746.55        19.00         77.71
50%        244

In [8]:
# --- preparar dataset para modelado ---

# 1) variables temporales
daily["dia_num"] = daily["fecha"].dt.dayofweek
daily["mes"] = daily["fecha"].dt.month
daily["es_fin_semana"] = daily["fecha"].dt.dayofweek.isin([5, 6]).astype(int)
daily["es_lunes"] = (daily["dia_num"] == 0).astype(int)

# 2) lags y medias móviles
daily = daily.sort_values("fecha").reset_index(drop=True)

for lag in [1, 3, 7, 14, 30]:
    daily[f"lag_{lag}"] = daily["facturacion"].shift(lag)

for window in [3, 7, 14, 30]:
    daily[f"media_movil_{window}"] = daily["facturacion"].shift(1).rolling(window, min_periods=1).mean()

# 3) quitar filas con NaN por los lags
daily = daily.dropna().reset_index(drop=True)

# 4) target y features
X = daily[
    ["dia_num", "mes", "es_fin_semana", "es_lunes",
     "lag_1", "lag_3", "lag_7", "lag_14", "lag_30",
     "media_movil_3", "media_movil_7", "media_movil_14", "media_movil_30"]
]

y = daily["facturacion"]

print("Shape X:", X.shape)
print("Shape y:", y.shape)
print(X.head())

Shape X: (212, 13)
Shape y: (212,)
   dia_num  mes  es_fin_semana  es_lunes    lag_1    lag_3    lag_7   lag_14  \
0        3   11              0         0  2383.50  1584.59  1227.80  1292.07   
1        4   11              0         0  1795.95  1056.35  2382.30   953.40   
2        5   11              1         0  3745.30  2383.50  3340.15  1621.89   
3        6   11              1         0  3816.95  1795.95  4133.07  4127.70   
4        1   11              0         0  1229.10  3745.30  1584.59  4902.10   

    lag_30  media_movil_3  media_movil_7  media_movil_14  media_movil_30  
0  1339.33    1674.813333    2301.108571     2371.530714     2622.679333  
1  4498.50    1745.266667    2382.272857     2407.522143     2637.900000  
2  4323.00    2641.583333    2576.987143     2606.943571     2612.793333  
3  1866.75    3119.400000    2645.101429     2763.733571     2595.925000  
4  1701.25    2930.450000    2230.248571     2556.690714     2574.670000  


In [9]:
### 7) Dividir train/test con validación temporal

# Orden temporal: train = primeras n observaciones, test = últimas m.
# Se evita la fuga de información y se simula una predicción futura realista.

split_idx = int(len(daily) * 0.8)
X_train, X_test = X.iloc[:split_idx].copy(), X.iloc[split_idx:].copy()
y_train, y_test = y.iloc[:split_idx].copy(), y.iloc[split_idx:].copy()

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Train target range:', y_train.min(), '->', y_train.max())
print('Test target range:', y_test.min(), '->', y_test.max())


Train shape: (169, 13)
Test shape: (43, 13)
Train target range: 637.9 -> 6785.2
Test target range: 788.7 -> 5661.0


In [10]:
# 8) Entrenar modelos y comparar predicciones
# La división temporal ya está hecha para que el test represente un futuro real.

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except Exception:
    xgb = None
    XGB_AVAILABLE = False

# Modelos base con distintos perfiles: lineal, regularizado, árboles y boosting
models = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'RandomForest': RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=1,
        random_state=SEED,
        n_jobs=-1
    ),
    'ExtraTrees': ExtraTreesRegressor(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=1,
        random_state=SEED,
        n_jobs=-1
    ),
    'GradientBoosting': GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=SEED,
    ),
    'HistGradientBoosting': HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=8,
        max_leaf_nodes=31,
        random_state=SEED,
    )
}

if XGB_AVAILABLE:
    models['XGBoost'] = xgb.XGBRegressor(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=SEED,
        n_jobs=-1,
        objective='reg:squarederror'
    )

# Grid de ajuste ligero para los modelos más potentes
# Se hace con validación temporal (TimeSeriesSplit) para mantener la lógica correcta
param_grid = {
    'RandomForest': {
        'n_estimators': [200, 400],
        'max_depth': [None, 8, 12],
        'min_samples_leaf': [1, 2]
    },
    'ExtraTrees': {
        'n_estimators': [200, 400],
        'max_depth': [None, 8, 12],
        'min_samples_leaf': [1, 2]
    },
    'GradientBoosting': {
        'n_estimators': [200, 400],
        'learning_rate': [0.03, 0.05],
        'max_depth': [2, 3]
    },
    'XGBoost': {
        'n_estimators': [200, 400],
        'max_depth': [3, 6],
        'learning_rate': [0.03, 0.05],
        'subsample': [0.8, 1.0]
    }
}

# Tuning temporal y evaluación final sobre test
results = []
for name, model in models.items():
    if name in param_grid:
        search = GridSearchCV(
            estimator=model,
            param_grid=param_grid[name],
            cv=TimeSeriesSplit(n_splits=3),
            scoring='neg_root_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        search.fit(X_train, y_train)
        best_model = search.best_estimator_
        print(f'\n=== {name} - tuning ===')
        print('Mejores parámetros:', search.best_params_)
        print('RMSE CV:', round(np.sqrt(-search.best_score_), 4))
    else:
        best_model = model

    best_model.fit(X_train, y_train)
    y_pred = best_model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({
        'modelo': name,
        'mae': mae,
        'rmse': rmse,
        'r2': r2,
    })

    pred_df = pd.DataFrame({
        'fecha': daily.loc[split_idx:split_idx + len(X_test) - 1, 'fecha'].values,
        'real': y_test.values,
        'prediccion': y_pred,
    })

    print(f'\n=== {name} ===')
    print(f'MAE:  {mae:.2f}')
    print(f'RMSE: {rmse:.2f}')
    print(f'R²:   {r2:.3f}')
    print(pred_df.head().to_string(index=False))
    print('...')
    print(pred_df.tail().to_string(index=False))

results_df = pd.DataFrame(results)
print('\nResumen comparativo:')
print(results_df.sort_values('rmse').to_string(index=False))



=== LinearRegression ===
MAE:  827.36
RMSE: 955.33
R²:   0.351
     fecha    real  prediccion
2026-05-21 1894.85 3111.071312
2026-05-22 5390.75 4705.760801
2026-05-23 4309.10 4432.260241
2026-05-24 2333.70 3017.537251
2026-05-26 1965.15  993.975873
...
     fecha    real  prediccion
2026-07-04 3251.55 3910.312980
2026-07-05 1728.35 2781.470384
2026-07-07 2119.25 1560.641114
2026-07-08 2183.20 2901.961293
2026-07-09  788.70 3159.686636

=== Ridge ===
MAE:  828.21
RMSE: 955.53
R²:   0.351
     fecha    real  prediccion
2026-05-21 1894.85 3112.653083
2026-05-22 5390.75 4702.003789
2026-05-23 4309.10 4436.493107
2026-05-24 2333.70 3016.843533
2026-05-26 1965.15  999.036164
...
     fecha    real  prediccion
2026-07-04 3251.55 3915.581904
2026-07-05 1728.35 2781.923947
2026-07-07 2119.25 1567.733511
2026-07-08 2183.20 2906.375076
2026-07-09  788.70 3159.877177

=== RandomForest - tuning ===
Mejores parámetros: {'max_depth': 8, 'min_samples_leaf': 2, 'n_estimators': 400}
RMSE CV: 33.7912

=

In [11]:
### 10) Validación temporal

cv = TimeSeriesSplit(n_splits=3)
print('\nTimeSeriesSplit aplicado con 3 folds:')
for i, (tr_idx, val_idx) in enumerate(cv.split(X_train), start=1):
    print(f'Fold {i}: train={len(tr_idx)}, val={len(val_idx)}')

print('\nLa validación temporal asegura que el modelo se evalúa con datos futuros reales.')



TimeSeriesSplit aplicado con 3 folds:
Fold 1: train=43, val=42
Fold 2: train=85, val=42
Fold 3: train=127, val=42

La validación temporal asegura que el modelo se evalúa con datos futuros reales.


In [12]:
### 13) Guardar modelo, métricas y resultados para presentación

# Guardar modelos y resultados en una carpeta de resultados del proyecto
# Para una primera versión, se guardan métricas y un resumen visual.

# Crear DataFrame de resultados si existe
if 'results_df' in globals():
    summary_path = PROJECT_ROOT / 'results' / 'forecasting_summary.csv'
    results_df.to_csv(summary_path, index=False)
    print(f'Métricas guardadas en: {summary_path}')

# Guardar también un resumen con predicciones del mejor modelo si existe
if 'results' in globals() and results:
    pd.DataFrame(results).to_csv(PROJECT_ROOT / 'results' / 'forecasting_model_summary.csv', index=False)
    print('Resumen de modelos guardado.')

if 'advanced_df' in globals() and not advanced_df.empty:
    adv_path = PROJECT_ROOT / 'results' / 'forecasting_advanced_summary.csv'
    advanced_df.to_csv(adv_path, index=False)
    print(f'Métricas avanzadas guardadas en: {adv_path}')

# nota: si quieres dejar este notebook como base de TFM, puedes guardar también el mejor modelo
# con joblib o pickle. Aquí se deja preparado para esa última etapa.
print('Notebook preparado para la predicción de facturación diaria.')


Métricas guardadas en: c:\Users\CandelaGB\Desktop\TFM anita\TFM-Hosteleria-AI\results\forecasting_summary.csv
Resumen de modelos guardado.
Notebook preparado para la predicción de facturación diaria.
